# 08 · Real options: the revenue-synergy expansion option

This upgrades the original illustrative notebook (`MA_Real_Options_Valuation.ipynb`, unchanged) with Disney–21CF data.
The Black-Scholes, CRR and GBM functions are reused from that notebook (`src/valuation/options.py`) and extended with a
continuous leakage yield *q*.

**The option.** The $2.0bn of cost synergies are committed and valued as cash flows (Stage 4). Revenue synergies are a
*right, not an obligation*: Disney's advisors valued them (May 2018 proxy p.145), but Disney never quantified them, and
capturing them needs further investment. They are modelled here as a call option to expand.

| | Definition | Base | Type |
|---|---|---|---|
| S | PV of revenue-synergy cash flows | advisors' implied $9.625/share × 1,871.1m ≈ $18.0bn | derived from disclosed ranges |
| K | PV of the investment required | = S (at the money: zero static NPV) | estimate |
| σ | volatility of the underlying | median peer asset volatility (~13.8%) | derived from market data |
| T | decision window | 3 years (to fiscal 2021) | estimate |
| r | risk-free | 3-year Treasury, 30 May 2018 (2.53%) | market |
| q | value leakage while waiting | 0% (base), 3% (American-exercise case) | estimate |

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 130); pd.set_option("display.max_rows", 300)
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
PROC, FIG = ROOT / "data" / "processed", ROOT / "docs" / "figures"
res = subprocess.run([sys.executable, str(ROOT / "src" / "valuation" / "build_stage5.py")], cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
print(res.stdout.strip()); assert res.returncode == 0, res.stdout + res.stderr
V = pd.read_csv(PROC / "verification_checks.csv")
RI = pd.read_csv(PROC / "real_options_inputs.csv"); RO = pd.read_csv(PROC / "real_options_outputs.csv"); BC = pd.read_csv(PROC / "binomial_convergence.csv")

In [ ]:
RI[["parameter", "value", "unit", "value_type", "source", "method", "sensitivity_low", "sensitivity_high"]]

## Black-Scholes, CRR binomial and Monte Carlo

In [ ]:
RO[RO.section.isin(["Option value", "Binomial cross-check", "Monte Carlo cross-check"])][["section", "case", "metric", "value", "notes"]]

In [ ]:
display(BC.round(4)); display(Image(FIG / "09_option_binomial_convergence.png"))

## Scenarios and sensitivities

In [ ]:
RO[RO.section.isin(["Scenario", "Sensitivity (Black-Scholes)"])][["section", "case", "metric", "value", "notes"]]

## What the option adds to the deal justification (amended $38 offer)

In [ ]:
RO[RO.section == "Deal justification with option"][["case", "value", "notes"]]

In [ ]:
v = V[V.check_type.str.startswith("realoption_")]; print(v.groupby(["check_type", "status"]).size()); assert (v.status == "pass").all()